In [602]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [603]:
# fetch data 

mco_syr_sw_data = pd.read_csv('flight_data\mco_syr_sw_22_23.csv')
mco_syr_jb_data = pd.read_csv('flight_data\mco_syr_jb_22_23.csv')
jfk_syr_jb_data = pd.read_csv('flight_data\jfk_syr_jb_22_23.csv')
jfk_syr_end_data = pd.read_csv('flight_data\jfk_syr_end_22_23.csv')
ord_syr_ua_data = pd.read_csv('flight_data\ord_syr_ua_22_23.csv')
ord_syr_aa_data = pd.read_csv('flight_data\ord_syr_aa_22_23.csv')
# mco_syr_sw_data.head()
# mco_syr_jb_data.head()
# jfk_syr_jb_data.head()
# jfk_syr_end_data.head()
# ord_syr_ua_data.head()
# ord_syr_aa_data.head()

In [604]:
dfs = [ord_syr_aa_data,ord_syr_ua_data,jfk_syr_end_data,jfk_syr_jb_data,mco_syr_jb_data,mco_syr_sw_data]
main_data = pd.concat(dfs,axis = 0)
main_data.head()
len(main_data)

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,0,MQ,2022-01-01,4316,N900AE,SYR,15:28,0:00,107,0,...,0:00,0,0,0,0,0,0,18,5,2022
1,1,MQ,2023-01-01,3392,N283NN,SYR,18:25,18:24,111,104,...,21:02,6,0,0,0,0,0,21,6,2023
2,2,MQ,2023-01-01,3518,N213NN,SYR,13:05,13:01,117,96,...,15:33,4,0,0,0,0,0,16,6,2023
3,3,MQ,2022-01-02,4134,N900AE,SYR,10:25,10:21,111,146,...,13:43,4,0,0,31,0,0,13,6,2022
4,4,MQ,2022-01-02,4253,N238NN,SYR,17:25,17:51,107,116,...,20:40,7,0,26,9,0,0,20,6,2022


5623

In [605]:
main_data.dtypes



Unnamed: 0                             int64
Carrier_Code                          object
Date                                  object
Flight_Number                          int64
Tail_Number                           object
Destination_Airport                   object
Scheduled departure time              object
Actual departure time                 object
Scheduled elapsed time (Minutes)       int64
Actual elapsed time (Minutes)          int64
Departure delay (Minutes)              int64
Wheels-off time                       object
Taxi-Out time (Minutes)                int64
dep_Delay_Carrier                      int64
dep_Delay_Weather                      int64
dep_Delay_National_Aviation_System     int64
dep_Delay_Security                     int64
dep_Delay_Late_Aircraft_Arrival        int64
dep_hour                               int64
dep_day                                int64
dep_year                               int64
dep_order                             object
Origin_Air

In [606]:
# main_data['dep_min'] = main_data['Scheduled departure time'].str.split(":").str[1].astype('int64')
# main_data['Date'] = pd.to_datetime( main_data['Date'],format ="%Y-%m-%d") 
# main_data['Scheduled departure time'] = pd.to_datetime( main_data['Scheduled departure time'],format ='%H:%M') 
# main_data['Actual Arrival Time'] = pd.to_datetime( main_data['Actual Arrival Time'],format ='%H:%M')
# #main_data['dep_day'] = main_data['Date'].dt.day_of_week.astype('int64') 
# #main_data['dep_year'] = main_data['Date'].dt.year.astype('int64')
# main_data.dtypes

In [607]:
sub_data = main_data.drop(columns = ['Unnamed: 0','Destination_Airport','Actual departure time','Scheduled elapsed time (Minutes)','Actual elapsed time (Minutes)',
                         'Departure delay (Minutes)','Wheels-off time','Taxi-Out time (Minutes)','dep_Delay_Carrier',
                         'dep_Delay_Weather','dep_Delay_National_Aviation_System','dep_Delay_Security','dep_Delay_Late_Aircraft_Arrival',
                         'dep_year','Actual Arrival Time','Wheels-on Time','Taxi-In time (Minutes)','arr_Delay_Carrier','arr_Delay_Weather',
                         'arr_Delay_National_Aviation_System','arr_Delay_Security','arr_Delay_Late_Aircraft_Arrival','arr_year'])

In [608]:
sub_data.dtypes


Carrier_Code                object
Date                        object
Flight_Number                int64
Tail_Number                 object
Scheduled departure time    object
dep_hour                     int64
dep_day                      int64
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                     int64
arr_day                      int64
dtype: object

In [609]:
sub_data.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,0,18,5
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,-8,21,6
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,-25,16,6
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,31,13,6
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,35,20,6


In [610]:
sub_data['dep_min'] = sub_data['Scheduled departure time'].str.split(":").str[1].astype('int64')
def round_to_nearest_quarter(minute):
    if 0 <= minute < 14:
        return 0
    elif 15 <= minute < 29:
        return 15
    elif 30 <= minute < 44:
        return 30
    else:
        return 45

sub_data['dep_minutes'] = sub_data['dep_min'].apply(round_to_nearest_quarter).astype('object')
sub_data['dep_hours'] = sub_data['dep_hour'].astype('object')
sub_data['dep_min'] = sub_data['dep_min'].astype('object')
sub_data['arr_min'] = main_data['Scheduled Arrival Time'].str.split(":").str[1].astype('int64')
sub_data['arr_minutes'] = sub_data['arr_min'].apply(round_to_nearest_quarter).astype('object')
sub_data['arr_hours'] = sub_data['arr_hour'].astype('object')
sub_data['arr_min'] = sub_data['arr_min'].astype('object')
sub_data['Flight_Number'] = main_data['Flight_Number'].astype('object')
sub_data['dep_hour'] = main_data['dep_hour'].astype('object')
sub_data['dep_day'] = main_data['dep_day'].astype('object')
sub_data['arr_hour'] = main_data['arr_hour'].astype('object')
sub_data['arr_day'] = main_data['arr_day'].astype('object')
conditions = [
    (sub_data['Arrival Delay (Minutes)'] > 5),
    (sub_data['Arrival Delay (Minutes)'] >=-5) & (sub_data['Arrival Delay (Minutes)'] <= 5),
    (sub_data['Arrival Delay (Minutes)'] < -5)
]
choices = [2,1,0]
sub_data['arr_status'] = np.select(conditions, choices)
sub_data.dtypes
sub_data.head()

Carrier_Code                object
Date                        object
Flight_Number               object
Tail_Number                 object
Scheduled departure time    object
dep_hour                    object
dep_day                     object
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                    object
arr_day                     object
dep_min                     object
dep_minutes                 object
dep_hours                   object
arr_min                     object
arr_minutes                 object
arr_hours                   object
arr_status                   int32
dtype: object

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day,dep_min,dep_minutes,dep_hours,arr_min,arr_minutes,arr_hours,arr_status
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,0,18,5,28,15,15,15,15,18,1
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,-8,21,6,25,15,18,16,15,21,0
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,-25,16,6,5,0,13,2,0,16,0
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,31,13,6,25,15,10,16,15,13,2
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,35,20,6,25,15,17,12,0,20,2


In [611]:
sub_data.to_csv('sub_data.csv', index=False)

In [612]:
# Read and process weather data files for each airport
jfk_weather_data = pd.read_csv('weather_data/JFK_weather_data_sub_hourly_processed.csv')
syr_weather_data = pd.read_csv('weather_data/SYR_weather_data_sub_hourly_processed.csv')
ord_weather_data = pd.read_csv('weather_data/ORD_weather_data_sub_hourly_processed.csv')
mco_weather_data = pd.read_csv('weather_data/MCO_weather_data_sub_hourly_processed.csv')

# Combine weather data for all airports
weather_dfs = [jfk_weather_data, ord_weather_data, mco_weather_data]
weather_data = pd.concat(weather_dfs, axis=0)

# Define merging logic based on airport code
sub_data = pd.merge(sub_data, syr_weather_data, how='left', on=['Date', 'arr_hours', 'arr_minutes'])

sub_data = pd.merge(sub_data, weather_data, how='left', on=['Origin_Airport', 'Date', 'dep_hours', 'dep_minutes'])


sub_data.head()


C:\Users\simra\AppData\Local\Temp\ipykernel_21076\315742201.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  jfk_weather_data = pd.read_csv('weather_data/JFK_weather_data_sub_hourly_processed.csv')


,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,...,dep_pres,dep_rh,dep_snow_rate,dep_temp,dep_vis,dep_weather.description,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,...,984,87,17.00,-0.80,1,600,s01d,30,13.40,10.55
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,...,990,86,0.00,4.40,8,804,c04n,215,3.20,0.86
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,...,989,88,0.00,5.60,5,804,c04d,230,6.40,3.60
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,...,996,77,0.00,-6.10,11,804,c04d,340,8.90,4.09
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,...,999,69,0.00,-5.30,16,802,c02n,325,5.40,3.09


In [613]:
sub_data.isna().sum()

Carrier_Code                 0
Date                         0
Flight_Number                0
Tail_Number                 11
Scheduled departure time     0
dep_hour                     0
dep_day                      0
dep_order                    0
Origin_Airport               0
Scheduled Arrival Time       0
Arrival Delay (Minutes)      0
arr_hour                     0
arr_day                      0
dep_min                      0
dep_minutes                  0
dep_hours                    0
arr_min                      0
arr_minutes                  0
arr_hours                    0
arr_status                   0
arr_clouds                   0
arr_dewpt                    0
arr_precip_rate              0
arr_pres                     0
arr_rh                       0
arr_snow_rate                0
arr_temp                     0
arr_vis                      0
arr_weather.description      0
arr_weather.code             0
arr_wind_dir                 0
arr_wind_gust_spd            0
arr_wind

In [614]:
sub_data.columns

Index(['Carrier_Code', 'Date', 'Flight_Number', 'Tail_Number',
       'Scheduled departure time', 'dep_hour', 'dep_day', 'dep_order',
       'Origin_Airport', 'Scheduled Arrival Time', 'Arrival Delay (Minutes)',
       'arr_hour', 'arr_day', 'dep_min', 'dep_minutes', 'dep_hours', 'arr_min',
       'arr_minutes', 'arr_hours', 'arr_status', 'arr_clouds', 'arr_dewpt',
       'arr_precip_rate', 'arr_pres', 'arr_rh', 'arr_snow_rate', 'arr_temp',
       'arr_vis', 'arr_weather.description', 'arr_weather.code',
       'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd', 'dep_clouds',
       'dep_dewpt', 'dep_precip_rate', 'dep_pres', 'dep_rh', 'dep_snow_rate',
       'dep_temp', 'dep_vis', 'dep_weather.description', 'dep_weather.code',
       'dep_wind_dir', 'dep_wind_gust_spd', 'dep_wind_spd'],
      dtype='object')

In [615]:
sub_data.to_csv('merged_data.csv', index=False)


In [616]:
sub_data.rename(columns={'Arrival Delay (Minutes)':'arr_delay'}, inplace=True)


In [617]:
sub_data.drop(columns= ['Date','Flight_Number','Tail_Number',
                       'Scheduled departure time','Scheduled Arrival Time','dep_order', 'arr_hours','arr_minutes', 'dep_hours', 'dep_minutes', 'arr_weather.description', 'dep_weather.description', 'arr_weather.code', 'dep_weather.code'],inplace = True)


In [618]:

sub_data.dtypes

sub_data.head()

Carrier_Code          object
dep_hour              object
dep_day               object
Origin_Airport        object
arr_delay              int64
arr_hour              object
arr_day               object
dep_min               object
arr_min               object
arr_status             int32
arr_clouds             int64
arr_dewpt            float64
arr_precip_rate      float64
arr_pres               int64
arr_rh                 int64
arr_snow_rate        float64
arr_temp             float64
arr_vis                int64
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_precip_rate      float64
dep_pres               int64
dep_rh                 int64
dep_snow_rate        float64
dep_temp             float64
dep_vis                int64
dep_wind_dir           int64
dep_wind_gust_spd    float64
dep_wind_spd         float64
dtype: object

,Carrier_Code,dep_hour,dep_day,Origin_Airport,arr_delay,arr_hour,arr_day,dep_min,arr_min,arr_status,...,dep_dewpt,dep_precip_rate,dep_pres,dep_rh,dep_snow_rate,dep_temp,dep_vis,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,MQ,15,5,ORD,0,18,5,28,15,1,...,-2.70,1.00,984,87,17.00,-0.80,1,30,13.40,10.55
1,MQ,18,6,ORD,-8,21,6,25,16,0,...,2.30,0.00,990,86,0.00,4.40,8,215,3.20,0.86
2,MQ,13,6,ORD,-25,16,6,5,2,0,...,3.80,0.00,989,88,0.00,5.60,5,230,6.40,3.60
3,MQ,10,6,ORD,31,13,6,25,16,2,...,-9.50,0.00,996,77,0.00,-6.10,11,340,8.90,4.09
4,MQ,17,6,ORD,35,20,6,25,12,2,...,-10.10,0.00,999,69,0.00,-5.30,16,325,5.40,3.09


In [619]:
sub_data['Carrier_Code'].unique()

array(['MQ', 'UA', '9E', 'B6', 'WN'], dtype=object)

In [620]:
sub_data["Carrier_Code"] = np.where(sub_data["Carrier_Code"] == "MQ", "AA", sub_data["Carrier_Code"])
sub_data["Carrier_Code"] = np.where(sub_data["Carrier_Code"] == "9E", "DL", sub_data["Carrier_Code"])

In [621]:
sub_data['Origin_Airport'].unique()

array(['ORD', 'JFK', 'MCO'], dtype=object)

In [622]:
su_data = sub_data
# su_data.head()
# sub_data.columns
su_data['dep_hour'] = pd.Categorical(su_data['dep_hour'], categories=[i for i in range(24)])
su_data['dep_day'] = pd.Categorical(su_data['dep_day'], categories=[i for i in range(7)])
su_data['dep_min'] = pd.Categorical(su_data['dep_min'], categories=[i for i in range(60)])
su_data['arr_hour'] = pd.Categorical(su_data['arr_hour'], categories=[i for i in range(24)])
su_data['arr_day'] = pd.Categorical(su_data['arr_day'], categories=[i for i in range(7)])
su_data['arr_min'] = pd.Categorical(su_data['arr_min'], categories=[i for i in range(60)])
su_data['Carrier_Code'] = pd.Categorical(su_data['Carrier_Code'], categories=['AA', 'UA', 'DL', 'B6', 'WN'])
su_data['Origin_Airport'] = pd.Categorical(su_data['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])
# su_data['arr_weather.code'] = pd.Categorical(su_data['arr_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
# su_data['dep_weather.code'] = pd.Categorical(su_data['dep_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
su_data.columns

Index(['Carrier_Code', 'dep_hour', 'dep_day', 'Origin_Airport', 'arr_delay',
       'arr_hour', 'arr_day', 'dep_min', 'arr_min', 'arr_status', 'arr_clouds',
       'arr_dewpt', 'arr_precip_rate', 'arr_pres', 'arr_rh', 'arr_snow_rate',
       'arr_temp', 'arr_vis', 'arr_wind_dir', 'arr_wind_gust_spd',
       'arr_wind_spd', 'dep_clouds', 'dep_dewpt', 'dep_precip_rate',
       'dep_pres', 'dep_rh', 'dep_snow_rate', 'dep_temp', 'dep_vis',
       'dep_wind_dir', 'dep_wind_gust_spd', 'dep_wind_spd'],
      dtype='object')

In [623]:
su_data = pd.get_dummies(su_data, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
su_data.head()
su_data.dtypes
su_data.columns

,arr_delay,arr_status,arr_clouds,arr_dewpt,arr_precip_rate,arr_pres,arr_rh,arr_snow_rate,arr_temp,arr_vis,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
0,0,1,100,6.00,1.25,991,95,0.00,6.70,8,...,False,False,False,False,False,False,False,False,False,False
1,-8,0,100,1.80,0.00,1000,88,0.00,3.60,12,...,False,False,False,False,False,False,False,False,False,False
2,-25,0,100,1.70,0.00,999,79,0.00,5.00,13,...,False,False,False,False,False,False,False,False,False,False
3,31,2,100,-6.20,0.00,994,91,0.00,-5.00,6,...,False,False,False,False,False,False,False,False,False,False
4,35,2,100,-5.60,0.00,1003,91,0.00,-4.40,16,...,False,False,False,False,False,False,False,False,False,False


arr_delay            int64
arr_status           int32
arr_clouds           int64
arr_dewpt          float64
arr_precip_rate    float64
                    ...   
arr_min_55            bool
arr_min_56            bool
arr_min_57            bool
arr_min_58            bool
arr_min_59            bool
Length: 206, dtype: object

Index(['arr_delay', 'arr_status', 'arr_clouds', 'arr_dewpt', 'arr_precip_rate',
       'arr_pres', 'arr_rh', 'arr_snow_rate', 'arr_temp', 'arr_vis',
       ...
       'arr_min_50', 'arr_min_51', 'arr_min_52', 'arr_min_53', 'arr_min_54',
       'arr_min_55', 'arr_min_56', 'arr_min_57', 'arr_min_58', 'arr_min_59'],
      dtype='object', length=206)

In [624]:
X_train, X_test, y_train, y_test = train_test_split(su_data.drop(columns = ['arr_delay']), su_data['arr_delay'], test_size=0.2, random_state=50)
# In the above split the stratify = y essentially makes sure the fractions of the classification is maintained
X_train
X_test
y_train.dtypes
y_test

,arr_status,arr_clouds,arr_dewpt,arr_precip_rate,arr_pres,arr_rh,arr_snow_rate,arr_temp,arr_vis,arr_wind_dir,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
2454,1,50,2.00,0.00,999,62,0.00,8.90,16,165,...,False,False,False,False,False,False,False,False,False,False
660,2,80,6.70,0.00,1008,83,0.00,9.40,16,160,...,False,False,False,False,False,False,False,False,False,False
2387,1,25,-8.50,0.00,1006,37,0.00,5.00,16,290,...,False,False,False,False,False,False,True,False,False,False
4058,1,93,0.00,0.00,1005,24,0.00,21.40,16,280,...,False,False,False,False,False,False,False,False,False,False
5085,2,100,3.90,0.00,997,44,0.00,16.10,16,90,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,1,100,-1.40,0.00,991,66,0.00,4.40,16,290,...,False,False,False,False,False,False,False,False,False,False
132,0,50,6.60,0.00,1002,64,0.00,13.30,16,270,...,False,False,False,False,False,False,False,False,False,False
2014,0,100,-5.10,0.00,1012,74,0.00,-1.10,14,260,...,False,False,False,False,False,False,False,False,False,False
1931,2,87,-18.40,0.00,1007,37,0.00,-6.20,16,70,...,False,False,False,False,False,False,False,False,False,False


,arr_status,arr_clouds,arr_dewpt,arr_precip_rate,arr_pres,arr_rh,arr_snow_rate,arr_temp,arr_vis,arr_wind_dir,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
1512,1,50,19.00,1.00,1000,78,0.00,23.00,14,290,...,True,False,False,False,False,False,False,False,False,False
4829,2,87,-4.40,0.00,1005,68,0.00,0.80,16,100,...,False,False,False,False,False,False,False,False,False,False
3401,0,100,-1.80,4.00,1004,81,8.00,1.10,10,75,...,False,False,False,False,False,False,False,False,False,False
2670,2,0,5.60,0.00,1000,71,0.00,10.60,16,350,...,False,False,False,False,False,False,False,False,False,False
2564,0,87,17.00,0.00,1000,66,0.00,23.70,16,290,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184,2,100,13.00,0.75,992,70,0.00,18.60,16,20,...,False,False,False,False,False,False,False,False,False,True
210,0,50,7.10,0.00,1009,44,0.00,19.70,16,105,...,False,False,False,False,False,False,False,False,False,False
979,1,100,2.90,0.00,1007,86,0.00,5.00,12,80,...,False,False,False,False,False,False,False,False,False,False
5392,2,100,0.00,0.00,1006,67,0.00,5.60,5,110,...,False,False,False,False,False,False,True,False,False,False


dtype('int64')

1512      5
4829     83
3401    -14
2670     54
2564    -10
       ... 
1184    161
210     -14
979      -1
5392      6
2623    -26
Name: arr_delay, Length: 1125, dtype: int64

In [625]:
y = X_train['arr_status']
u = X_test['arr_status']
X_train.drop(columns=['arr_status'], inplace=True)
X_test.drop(columns=['arr_status'], inplace=True)

In [626]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = pd.DataFrame(sc.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(sc.transform(X_test), columns = X_test.columns, index = X_test.index)
X_train
X_test
y_train
y_test

,arr_clouds,arr_dewpt,arr_precip_rate,arr_pres,arr_rh,arr_snow_rate,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
2454,-0.87,-0.25,-0.11,-0.20,-0.07,-0.11,-0.28,0.40,-0.47,-1.43,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
660,0.15,0.26,-0.11,0.96,1.08,-0.11,-0.23,0.40,-0.53,-1.31,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
2387,-1.71,-1.37,-0.11,0.70,-1.44,-0.11,-0.67,0.40,0.93,0.04,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,5.66,-0.11,-0.13,-0.27
4058,0.59,-0.46,-0.11,0.57,-2.16,-0.11,0.96,0.40,0.81,0.89,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
5085,0.82,-0.04,-0.11,-0.46,-1.06,-0.11,0.43,0.40,-1.31,-0.55,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,0.82,-0.61,-0.11,-1.23,0.15,-0.11,-0.73,0.40,0.93,1.95,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
132,-0.87,0.24,-0.11,0.19,0.04,-0.11,0.15,0.40,0.70,0.92,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
2014,0.82,-1.00,-0.11,1.48,0.59,-0.11,-1.28,-0.22,0.59,0.68,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
1931,0.38,-2.42,-0.11,0.83,-1.44,-0.11,-1.78,0.40,-1.53,0.19,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27


,arr_clouds,arr_dewpt,arr_precip_rate,arr_pres,arr_rh,arr_snow_rate,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
1512,-0.87,1.57,0.42,-0.07,0.81,-0.11,1.12,-0.22,0.93,0.74,...,8.53,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
4829,0.38,-0.93,-0.11,0.57,0.26,-0.11,-1.09,0.40,-1.20,0.27,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
3401,0.82,-0.65,2.02,0.44,0.97,3.26,-1.06,-1.46,-1.48,2.39,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
2670,-2.56,0.14,-0.11,-0.07,0.42,-0.11,-0.11,0.40,1.60,0.80,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
2564,0.38,1.35,-0.11,-0.07,0.15,-0.11,1.19,0.40,0.93,-1.25,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,5.66,-0.11,-0.13,-0.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184,0.82,0.93,0.29,-1.10,0.37,-0.11,0.68,0.40,-2.09,-0.14,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,3.69
210,-0.87,0.30,-0.11,1.09,-1.06,-0.11,0.79,0.40,-1.14,-0.61,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
979,0.82,-0.15,-0.11,0.83,1.25,-0.11,-0.67,-0.84,-1.42,0.21,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,-0.18,-0.11,-0.13,-0.27
5392,0.82,-0.46,-0.11,0.70,0.20,-0.11,-0.61,-3.00,-1.09,-0.64,...,-0.12,-0.17,-0.01,-0.16,-0.09,-0.12,5.66,-0.11,-0.13,-0.27


2454     -2
660      29
2387     -4
4058     -5
5085    112
       ... 
70       -3
132     -12
2014    -23
1931     14
5600      8
Name: arr_delay, Length: 4498, dtype: int64

1512      5
4829     83
3401    -14
2670     54
2564    -10
       ... 
1184    161
210     -14
979      -1
5392      6
2623    -26
Name: arr_delay, Length: 1125, dtype: int64

In [627]:
#model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'ovr', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'multinomial', penalty = None, max_iter = 1000)
model = LinearRegression(fit_intercept = True)
# model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 10000)

# While using multiclass case do multi_class = 'ovr' or 'auto'; can also try other solvers
# While doing regularization, use penalty = 'l2' and also C = 10.0 (need to try other values too)

model.fit(X_train, y_train) 

# The following gives the mean accuracy on the given data and labels
model.score(X_train, y_train) 

# This is the coefficient Beta_1, ..., Beta_7
model.coef_

# This is the coefficient Beta_0
model.intercept_

LinearRegression()

0.18480729984181632

array([ 9.56827970e-01,  5.01627525e+00, -3.23797934e-01,  9.65590334e-01,
       -1.50863725e+00, -8.57720364e-01, -3.26668398e+00, -3.93712029e+00,
        1.53942798e+00, -1.69030535e+00,  3.65228065e+00,  8.99552771e-01,
        1.12194614e+01,  1.26344673e+01, -3.00116407e+00, -4.15991495e+00,
       -4.53574886e-01, -9.75909692e+00, -4.68469117e+00,  2.19045891e+00,
        6.19199391e+00, -3.39642885e+00,  4.84205154e+00,  1.90418069e+13,
        1.92821180e+13,  7.17353963e+12, -1.26323826e+14,  1.23999194e+14,
        1.38448586e+14, -4.94797088e+14,  2.65170321e+14,  3.85633678e+14,
       -1.04833375e+13, -9.85975450e+13, -1.12659703e+14, -6.40927269e+13,
       -7.67761661e+13, -7.85209845e+13, -1.43409834e+14, -9.26817872e+13,
       -8.08481231e+13, -1.22322851e+14, -4.73440828e+13, -1.24096390e+14,
       -6.07396306e+13, -6.13125453e+13, -1.14795769e+14, -1.46421400e+14,
       -1.71096946e+13, -4.92431866e+14, -5.87544724e+14,  6.54060223e+13,
       -1.34418760e+14,  

10.460712842843833

In [628]:
model.score(X_test,y_test)


-2.178511580769069e+23

In [629]:
from sklearn.tree import DecisionTreeClassifier
# from sklearn.tree import DecisionTreeRegressor
clf = DecisionTreeClassifier(random_state=50, min_samples_leaf = 3)

clf = clf.fit(X_train, y_train) 
clf.score(X_train, y_train)

clf.feature_importances_
test_output = pd.DataFrame(clf.predict(X_test), index = X_test.index, columns = ['pred_arr_status'])
test_output.head()
clf.score(X_test, y_test)

0.49533125833703867

array([0.0354385 , 0.0608378 , 0.00349943, 0.05204268, 0.05399183,
       0.        , 0.05198558, 0.00809083, 0.04658834, 0.06580238,
       0.06005018, 0.03584087, 0.04938394, 0.00124775, 0.05258961,
       0.06929822, 0.00063641, 0.05320489, 0.00350851, 0.04586091,
       0.05582513, 0.05612633, 0.00413324, 0.00496119, 0.00445346,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.00190689, 0.00082085,
       0.        , 0.0014176 , 0.00070225, 0.00133526, 0.        ,
       0.00378341, 0.00220976, 0.        , 0.00475035, 0.00070225,
       0.00078205, 0.00140184, 0.00536299, 0.        , 0.0014045 ,
       0.00124482, 0.00315951, 0.00049078, 0.00370971, 0.00246289,
       0.00158006, 0.00105337, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00087437, 0.0009217 , 0.        , 0.00085755, 0.0009217 ,
       0.00077386, 0.00166802, 0.00267733, 0.00127597, 0.     

,pred_arr_status
1512,-10
4829,-21
3401,-14
2670,-33
2564,-20


0.02311111111111111

In [630]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=50, min_samples_split = 12, min_samples_leaf = 6, max_depth = 4, n_estimators = 300)

gb = gb.fit(X_train, y_train) 
gb.score(X_train, y_train) 

KeyboardInterrupt: 

In [ ]:
gb.score(X_test,y_test)

0.5191111111111111